## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
%matplotlib inline

In [2]:
import os 
import sys
from pathlib import Path

In [10]:
import torch 
from torchvision import datasets
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
# import ...

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device = torch.device(device)
device

device(type='mps')

### Step-1: - Setup Datasets

In [5]:
def find_project_root(start=None, markers=("pyproject.toml", ".git", "requirements.txt", ".gitignore")):
    p = Path(start or Path.cwd()).resolve()
    for cur in [p, *p.parents]:
        if any((cur / m).exists() for m in markers):
            return cur
    return p
BASE_CODE_DIR_PATH = find_project_root()
DATASET_DIR = BASE_CODE_DIR_PATH / 'datasets'
DATASET_DIR, BASE_CODE_DIR_PATH

(PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/Reference and Learning Content/Deep-Learning/datasets'),
 PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/Reference and Learning Content/Deep-Learning'))

In [6]:
initial_transform= transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=0.5, std=0.5)
]
)

In [7]:
# TODO: load FashionMNIST train and test with ToTensor + Normalize(mean=0.5, std=0.5)

train_data = datasets.FashionMNIST(root= DATASET_DIR, download =True, train = True, transform= initial_transform)
test_data = datasets.FashionMNIST(root= DATASET_DIR, download =True, train = False, transform= initial_transform)

In [8]:
assert len(train_data) == 60000
assert len(test_data) == 10000
assert train_data[0][0].shape == (1, 28, 28)

In [9]:
# TODO: DataLoader, batch_size=128, shuffle train
BATCH_SIZE = 128
train_loader = DataLoader(train_data, shuffle=True, batch_size = BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=True, batch_size = BATCH_SIZE)

imgs, labels = next(iter(train_loader))
assert imgs.shape == (128, 1, 28, 28)
assert labels.shape == (128,)

## Shape math

In [ ]:
# TODO: fill in every shape as a comment BEFORE writing any model code
# input:             (B, 1, 28, 28)
# conv1(1->32, 3):   (B, ?, ?, ?)
# relu:              (B, ?, ?, ?)
# maxpool(2):        (B, ?, ?, ?)
# conv2(32->64, 3):  (B, ?, ?, ?)
# relu:              (B, ?, ?, ?)
# maxpool(2):        (B, ?, ?, ?)
# flatten:           (B, ?)
# linear(?, 128):    (B, 128)
# relu:              (B, 128)
# linear(128, 10):   (B, 10)

## Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels:int, num_classes:int):
        # TODO: define conv1, conv2, fc1, fc2, pool
        super(SimpleCNN, self).__init__()
        
        # self.conv1 = nn.Conv2d(in_channels= in_channels,
        #                        out_channels=64
        #                        )

    def forward(self, x):
        # TODO: implement forward, print shape after each layer (remove prints after verifying)
        ...

model = SimpleCNN()
out = model(imgs)
assert out.shape == (128, 10)

In [ ]:
# TODO: verify flatten size matches your shape math
# note: if mismatch, you have a shape bug — fix shape math first, then code
with torch.no_grad():
    x = torch.randn(1, 1, 28, 28)
    # TODO: run through conv layers only, print flattened size
    flatten_size = ...

assert flatten_size == model.fc1.in_features

## Training

In [ ]:
model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 10
warmup_epochs = 3
base_lr = 1e-3

for epoch in range(epochs):
    # TODO: lr warmup — linearly increase lr from 0 to base_lr over warmup_epochs
    if epoch < warmup_epochs:
        lr = ...
        for pg in optimizer.param_groups:
            pg["lr"] = lr

    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        # TODO: forward, loss, backward, step, zero_grad
        ...

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # TODO: accumulate correct
            ...

    print(f"epoch {epoch}: loss={total_loss/len(train_loader):.4f} acc={correct/total:.4f}")

assert correct / total > 0.80, f"accuracy too low: {correct/total}"

## Shape debugging

In [ ]:
# TODO: wrong padding exercise — set padding=0 on conv1, predict the new output shape
# expected shape after conv1 with padding=0: (B, 32, ?, ?)
# expected shape after pool1: (B, 32, ?, ?)
# expected shape after conv2 with padding=0: (B, 64, ?, ?)
# expected shape after pool2: (B, 64, ?, ?)
# expected flatten size: ?

class CNN_NoPad(nn.Module):
    def __init__(self):
        # TODO: same as SimpleCNN but padding=0
        ...
    def forward(self, x):
        ...

model_nopad = CNN_NoPad()
out_nopad = model_nopad(imgs)
assert out_nopad.shape == (128, 10)

In [ ]:
# TODO: stride=2 vs MaxPool — replace MaxPool2d(2) with stride=2 in conv layers
# predict shapes, verify they match MaxPool version

class CNN_Stride(nn.Module):
    def __init__(self):
        # TODO: conv1(1->32, 3, stride=2, pad=1), conv2(32->64, 3, stride=2, pad=1), no pool
        ...
    def forward(self, x):
        ...

model_stride = CNN_Stride()
out_stride = model_stride(imgs)
# TODO: fill in the expected shape — it differs from SimpleCNN
assert out_stride.shape == (128, 10)